# L13: SQL and grouped analytics

Use a local SQLite database for joins, CASE, CTEs, NULL, aggregation, and windows.

**Environment:** See ../requirements.txt and the labs index. Examples use bundled inputs; no internet is required after dependency installation. Synthetic data is for teaching only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "datasets").exists() else Path.cwd().parent
assert (ROOT / "datasets").exists(), "Run from the repository root or labs directory; in Colab upload/clone the repository first."
DATA = ROOT / "datasets"


## Same question in SQL and pandas

In [ ]:
import sqlite3
sales=pd.read_csv(DATA / "sales_multiyear.csv")
regions=pd.read_csv(DATA / "dim_region.csv")
con=sqlite3.connect(":memory:")
sales.to_sql("sales",con,index=False);regions.to_sql("regions",con,index=False)
query="""WITH regional AS (
 SELECT r.Region, SUM(s.NetRevenue) AS revenue,
 COUNT(DISTINCT s.OrderID) AS orders
 FROM sales s JOIN regions r ON s.RegionKey=r.RegionKey
 WHERE s.OrderDate >= '2024-01-01'
 GROUP BY r.Region
 HAVING SUM(s.NetRevenue) > 0
)
SELECT *, revenue/orders AS aov, RANK() OVER(ORDER BY revenue DESC) AS revenue_rank
FROM regional ORDER BY revenue_rank"""
print(pd.read_sql_query(query,con))
print(pd.read_sql_query("SELECT CASE WHEN Discount=0 THEN 'Full price' ELSE 'Discounted' END AS price_type, COUNT(*) AS lines FROM sales GROUP BY price_type",con))
print(pd.read_sql_query("SELECT COALESCE(NULL, 'Unknown') AS fallback, NULL IS NULL AS is_missing",con))

## Grouped versus pivot layout

In [ ]:
print(sales.groupby(["Region","Category"]).NetRevenue.sum().head())
pivot=pd.pivot_table(sales,index="Region",columns="Category",values="NetRevenue",aggfunc="sum",fill_value=0,margins=True)
assert np.isclose(pivot.loc["All","All"],sales.NetRevenue.sum())
print(pivot)

## Student task
Why can a join duplicate revenue? Explain SQL NULL versus zero. Add a monthly revenue CTE and a trailing three-row sum window.

## Reference solution
Try the task before reading this cell.

In [ ]:
q="""WITH monthly AS (SELECT substr(OrderDate,1,7) AS month,SUM(NetRevenue) AS revenue FROM sales GROUP BY month)
SELECT month,revenue,SUM(revenue) OVER(ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS trailing_3_months FROM monthly"""
print(pd.read_sql_query(q,con).head())

## Interpretation and limitations
Record the dataset grain, units, assumptions, and a limitation in your submission. Explain one result rather than only submitting screenshots.